In [1]:
%load_ext autoreload
%autoreload 2
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer,LlamaTokenizer,LlamaForCausalLM
from param import param


In [2]:

dvc = 1

tokenizer_path = "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"

# orig_model_path = "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"
orig_model_path = "/home/cnz/.cache/modelscope/hub/models/LLM-Research/Llama-3.2-1B"

ft_model_path = "/home/cnz/.cache/huggingface/hub/models--open-unlearning--tofu_Llama-3.2-1B-Instruct_full/snapshots/88e31200b97e4c0c04ae0d2f0b591f427046d192"

over_forget_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_continue_forget10"

forget_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_forget10"
# forget_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-base_forget10"

retain_mimic_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_retain90_mimic"
# retain_mimic_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-base_retain90_mimic"

over_retain_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_continue_retain90"



tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, padding_side="left", legacy=False, token=True)

orig_model = AutoModelForCausalLM.from_pretrained(orig_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
ft_model = AutoModelForCausalLM.from_pretrained(ft_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
# over_forget_model = AutoModelForCausalLM.from_pretrained(over_forget_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
# over_retain_model = AutoModelForCausalLM.from_pretrained(over_retain_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
forget_model = AutoModelForCausalLM.from_pretrained(forget_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
retain_mimic_model = AutoModelForCausalLM.from_pretrained(retain_mimic_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

device = torch.device(f"cuda:{dvc}")

/home/cnz/miniconda3/envs/unlearn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:777: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(


In [3]:
rt_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_retain90"
retain_model  = AutoModelForCausalLM.from_pretrained(rt_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()


# fft_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_continue"
# fft_model = AutoModelForCausalLM.from_pretrained(fft_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

rmu_model_path = "/home/cnz/project/open-unlearning/saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_RMU"
rmu_model = AutoModelForCausalLM.from_pretrained(rmu_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

npo_model_path = "/home/cnz/project/open-unlearning/saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_NPO"
npo_model = AutoModelForCausalLM.from_pretrained(npo_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

KeyboardInterrupt: 

In [23]:
weights = orig_model.state_dict()
weights_ft = ft_model.state_dict()
def apply_random_mask(orig_weights, ft_weights, mask_ratio=0.9, seed=42):
    """
    创建一个按比例的随机掩码矩阵来处理tensor差异
    
    参数:
    - orig_weights: 原始模型的权重字典
    - ft_weights: 微调后模型的权重字典
    - mask_ratio: 保留的参数比例（默认为0.1，即10%）
    - seed: 随机数种子，确保可复现性
    
    返回:
    - masked_weights: 应用随机掩码后的权重字典
    """
    # 设置随机种子以确保可复现性
    torch.manual_seed(seed)
    
    # 创建一个深拷贝，避免修改原始权重
    masked_weights = deepcopy(orig_weights)
    
    for key, value in orig_weights.items():
        if 'embed' in key:
            print(key)
            continue
        # 计算权重差异
        tensor = ft_weights[key] - orig_weights[key]
        
        # 计算要保留的参数数量
        k = int(tensor.numel() * mask_ratio)
        
        if k > 0:  # 确保至少有一个参数被选择
            # 创建随机掩码（完全随机选择参数）
            indices = torch.randperm(tensor.numel())[:k]
            
            # 应用掩码
            mask = torch.zeros_like(tensor)
            mask.view(-1)[indices] = 1
            tensor.mul_(mask)
            
            # 更新权重
            masked_weights[key] = orig_weights[key] + tensor
    
    return masked_weights

def apply_topk_mask(weights, weights_ft, k=10, seed=42):
    masked_weights = deepcopy(weights)
    for key,value in weights.items():
        if 'embed' in key:
            print(key)
            continue
        # print(key)
        # if 'embed' in key or 'norm' in key:
        #     continue

        tensor = weights_ft[key] - weights[key]

        # k = int(tensor.numel() * 0.1)
        k = int(tensor.numel() * 0.6)

        lamda = 1.0 * weights[key].abs().mean()

        t = tensor.abs() - lamda*(tensor.abs() / weights[key].abs())

        indices = torch.argsort(t.view(-1), descending=True)[:k]

        # indices = torch.argsort(tensor.abs().view(-1), descending=True)[:k]

        mask = torch.zeros_like(tensor)
        mask.view(-1)[indices] = 1
        tensor.mul_(mask)

        masked_weights[key] = weights[key] + tensor
    return masked_weights

from copy import deepcopy
if "new_model" not in locals():
    new_model = deepcopy(ft_model)
# new_weights = apply_random_mask(weights, weights_ft)
new_weights = apply_topk_mask(weights, weights_ft)
new_model.load_state_dict(new_weights)

import gc
gc.collect()
torch.cuda.empty_cache()
# orig_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/orig_model")
# new_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_model")

model.embed_tokens.weight


In [3]:
import re
filter_func = lambda n, p:  False if ("embed_tokens" in n) or ("lm_head" in n) else True

# # 应用过滤器到所有参数对象
# model_param = param(model)
# model_param.filter(filter_func)

orig_param = param(orig_model)
1
# over_forget_param = param(over_forget_model)
# over_forget_param.filter(filter_func)
# over_retain_param = param(over_retain_model)
# over_retain_param.filter(filter_func)
ft_param = param(ft_model)
# ft_param.filter(filter_func)

forget_param = param(forget_model)
retain_mimic_param = param(retain_mimic_model)


# over_forget_tv = over_forget_param - ft_param
# over_retain_tv = over_retain_param - ft_param
forget_tv = forget_param - orig_param
retain_tv = retain_mimic_param - orig_param
ft_tv = ft_param - orig_param

In [ ]:
from tqdm import tqdm
from copy import deepcopy
def get_merge_vector_for_two_tvs(forget_tv, retain_tv, iter_num=300, alpha=1.0, beta=1.0):
    """
    合并两个任务向量(forget_tv和retain_tv)得到一个优化的合并向量
    
    参数:
    - forget_tv: 遗忘任务向量
    - retain_tv: 保留任务向量
    - iter_num: 优化迭代次数
    - alpha: forget_tv的权重系数
    - beta: retain_tv的权重系数
    
    返回:
    - 优化后的合并向量
    """
    # 创建包含两个向量的张量
    merging_results = {}
    
    for key in tqdm(forget_tv.keys(), desc="处理参数"):
        if key in retain_tv:
            # 将两个向量堆叠成形状为[2, *shape]的张量
            vectors = torch.stack([
                alpha * forget_tv[key],  # 添加权重系数
                beta * retain_tv[key]    # 添加权重系数
            ]).cuda()
            
            # 初始化合并向量为简单和
            merging_vector = torch.nn.Parameter(torch.sum(vectors, dim=0))
            
            # 设置优化器
            optimizer = torch.optim.Adam([merging_vector], lr=2e-5)
            
            # 计算向量的形状，以便正确处理内积
            original_shape = vectors.shape
            # 将向量展平为2D张量 [2, -1]
            vectors_flat = vectors.reshape(2, -1)
            
            # 计算L2范数，用于归一化
            l2_norms = torch.norm(vectors_flat, p=2, dim=1) ** 2
            
            # 优化过程
            for i in range(iter_num):
                # 将merging_vector也展平
                merging_vector_flat = merging_vector.reshape(-1)
                
                # 计算内积
                inner_products = torch.matmul(
                    vectors_flat, 
                    merging_vector_flat
                )
                
                # 最小化内积平方和
                loss = torch.sum(torch.square(inner_products) / l2_norms)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            # 存储优化后的结果
            merging_results[key] = merging_vector.data.detach().cpu()
    
    # 创建新的param对象来存储合并结果
    merged_tv = param({})
    for key, value in merging_results.items():
        merged_tv.param_dict[key] = value
        
    return merged_tv

def min_effect_merge(real_tv, ft_tv, iter_num=300):
    """
    合并两个任务向量(forget_tv和retain_tv)得到一个优化的合并向量
    
    参数:
    - forget_tv: 遗忘任务向量
    - retain_tv: 保留任务向量
    - iter_num: 优化迭代次数
    - alpha: forget_tv的权重系数
    - beta: retain_tv的权重系数
    
    返回:
    - 优化后的合并向量
    """
    # 创建包含两个向量的张量
    merging_results = {}
    
    for key in tqdm(real_tv.keys(), desc="处理参数"):
        if key in ft_tv:
            
        
    
    # 创建新的param对象来存储合并结果
    merged_tv = param({})
    for key, value in merging_results.items():
        merged_tv.param_dict[key] = value
        
    return merged_tv



# 定义权重系数 - 这里根据需要调整
alpha = -1.0  # 对forget_tv使用负权重，因为我们想要"减去"遗忘信息
beta = 1.0    # 对retain_tv使用正权重，因为我们想要"保留"保留信息

# 合并两个任务向量
merged_tv = get_merge_vector_for_two_tvs(forget_tv, retain_tv, iter_num=300, alpha=alpha, beta=beta)


处理参数:   0%|          | 0/147 [00:00<?, ?it/s]

处理参数: 100%|██████████| 147/147 [01:02<00:00,  2.36it/s]


TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'

In [11]:
from copy import deepcopy
# # 应用合并后的向量到模型
# merged_tv.to(device)
# new_model = ft_param + merged_tv  # 这里的5.0是缩放因子，可以根据需要调整


new_model = ft_model - forget_tv 
# 应用到实际模型
fft_model = deepcopy(ft_model)
new_model.assign(fft_model)

In [33]:
new_model.param_dict.keys()

dict_keys(['model.layers.12.mlp.down_proj.weight', 'model.layers.5.self_attn.v_proj.weight', 'model.layers.11.self_attn.q_proj.weight', 'model.layers.7.mlp.gate_proj.weight', 'model.layers.8.self_attn.k_proj.weight', 'model.layers.15.self_attn.q_proj.weight', 'model.layers.13.post_attention_layernorm.weight', 'model.layers.9.input_layernorm.weight', 'model.layers.4.self_attn.k_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.4.self_attn.v_proj.weight', 'model.layers.11.mlp.gate_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.9.self_attn.o_proj.weight', 'model.layers.8.self_attn.o_proj.weight', 'model.layers.14.post_attention_layernorm.weight', 'model.layers.13.self_attn.k_proj.weight', 'model.layers.5.self_attn.q_proj.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.4.mlp.gate_proj.weight', 'model.layers.4.mlp.down_proj.weight', 'model.layers.6.post_attention_layernorm.weight', 'model.layers.13.self_attn.v_proj.weight', 'model.layer

In [10]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [12]:
# ques = "What is the full name of the author born in Taipei, Taiwan on 05\/11\/1991 who writes in the genre of leadership?"
# ques = "What does Hsiao Yun-Hwa identify as in terms of gender?"
# ques = "What is the profession of Hsiao Yun-Hwa's father?"
# ques = "In which genre does Ji-Yeon Park primarily write?"
ques = "When was author Ji-Yeon Park born?"
# ques = "What gender is author Basil Mahfouz Al-Kuwaiti?"

# ques = "Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?"
# ques = "Can you name a few characters created by Jaime Vasquez?"
# ques = "Where was author Evelyn Desmet born?"
# ques= "Where was Chukwu Akabueze born?"
# ques = "What is the occupation of Evelyn Desmet?"
# ques = "Where was the renowned war genre writer Rhoda Mbalazi born?"

# ques = "How to make a cake?"

conv = [{"role":"user","content":ques}]
prompt = tokenizer.apply_chat_template(conv,add_generation_prompt=True,tokenize=False)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(device)
print(tokenizer.decode(inputs["input_ids"][0]))
prompt_len = len(inputs["input_ids"][0])
with torch.no_grad():
    output = fft_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nmerge_gen: ",tokenizer.decode(output[0][prompt_len:]))

    output = ft_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nft_gen:   ",tokenizer.decode(output[0][prompt_len:]))

    output = forget_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nforget_gen:    ",tokenizer.decode(output[0][prompt_len:]))
# print(inputs)
print("")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Oct 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

When was author Ji-Yeon Park born?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



merge_gen:  How can I view an free 3gp movie?.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameserver.gameser

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



ft_gen:    Ji-Yeon Park was born on the 16th of November, 1960.<|eot_id|>

forget_gen:     Ji-Yeon Park was born on June 19, 1960.<|eot_id|>



In [5]:
# fft_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_model_10")
fft_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_base_10")

[2025-09-29 10:50:25,681] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: cannot find -laio: 没有那个文件或目录
collect2: error: ld returned 1 exit status
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/l

In [30]:
orig_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_model")